# Week 8 — Make the last step work: lower, let go, back off

**Goal: the fly brain completes pick-and-place consistently: at least 40 of the 50 test seeds (80%).**

## Where week 7 left off
Both networks (connectome and random, at matched 80 ms timing) lifted the cube and carried it to the zone on **50/50** seeds, but completed **0/50**. The arm hovered a median 19–33 mm from the target, cube still gripped, until time ran out. The week 4 expert only starts lowering once the cube is within **8 mm** of the target, a sharp cut-off in a distance the brain can't resolve that finely: the 8 signals are encoded on a 5 cm scale, so 8 mm is a small, noisy input change, and the readout's sideways accuracy was the weakest part (dx R² 0.27).

## Ideas considered
| idea | what it targets | cost | keeps "the brain did it"? | status |
|---|---|---|---|---|
| **B. Graded, press-down expert** (teacher changes, brain doesn't) | the 8 mm cliff, and the 6 mm release cliff | none | yes: a linear readout without a brain still completes 0/50 | **tested: the fix** |
| C. Fine goal encoding (goal − cube on a 2 cm scale, not 5 cm) | input resolution near the target | none | yes | tested: helps alone, adds variance with B; not used |
| D. Smooth the DN rates over ~2 steps before the readout | spike-count noise | none | yes (a fixed filter, like a synapse) | tested: hurt on 3 of 8 settings; not used |
| E. More DAgger rounds | the release is rare in the training data | ~10 min per round | yes | the fallback (`EXTRA_ROUNDS = 2`) |
| F. Phase-gated readouts (one linear readout per decoded phase) | the open → closed → open sequence | none | weaker: the gating adds the nonlinearity | held back |
| G. Add a "distance to target" input | gives the readout the answer directly | none | **no**: a weighted sum could then do it alone | rejected |
| H. Hard-wired release reflex ("once low, open and lift") | the last step itself | none | **no**: the script, not the brain, would do the last step | rejected |
| I. RL fine-tuning of the readout (PPO) | everything | days of GPU | yes | too slow (week 7) |

## What the sandbox showed (stand-in brain, real physics, 50 test seeds)
The stand-in: 1,332 rectified random units with Poisson spikes (~12 Hz, ~25% active, 80 ms window, like the real DNs), the same ridge readout and DAgger, with input noise and an optional one-step reporting delay. At input noise 0.08 it fails the way the connectome did (3/50, 34 episodes hovering ~20 mm off target). Results depend a lot on the noise setting and seed, so every fix was tested on 8 settings (noise 0.08–0.12, 3 seeds, delay 0 or 1 step).

**Two failures, found one after the other:**
1. **Hovering above the target** (week 7's): fixed by a **graded height target**: the gripper lowers gradually as the cube approaches, instead of only once within 8 mm.
2. **Stopping just above the table.** With hovering gone, the arm lowered but stopped with the cube gripped 1–2 cm up. The week 4 expert opens only once the cube is within 6 mm of the table, and its downward command shrinks to a few mm per step in the last centimetre, too small for the noisy readout. Fixed by **pressing down**: the placing target is 8 mm *below* the table, so the command stays clearly downward until the table stops the cube; the gripper opens once the cube is within 10 mm of the table.
3. **Then the ramp's width.** With the ramp ending at 10 mm, a readout whose xy settled ~2 cm off target hovered half-lowered (4/50 on one setting). The zone allows 50 mm, so the ramp now runs from 50 mm (start lowering) to 25 mm (fully lowered), with release allowed within 40 mm. The expert itself still places within ~3 mm, because xy keeps correcting while it lowers.

**Robustness grid: success on 50 test seeds, the same DAgger (4 × 30) for every cell.**

| stand-in setting (noise / seed / delay) | week 7 setup | narrow ramp + press | **wide ramp + press (fix B)** | B + smoothing (D) |
|---|---|---|---|---|
| 0.08 / 0 / 0 (reproduces week 7) | 3/50 | 50/50 | **50/50** | 50/50 |
| 0.08 / 1 / 0 | 50/50 | 50/50 | **50/50** | 50/50 |
| 0.08 / 2 / 1 | 48/50 | 50/50 | **49/50** | 48/50 |
| 0.10 / 0 / 1 | 34/50 | 44/50 | **50/50** | 43/50 |
| 0.10 / 1 / 1 | 48/50 | 50/50 | **50/50** | 50/50 |
| 0.10 / 2 / 0 | 16/50 | 4/50 | **48/50** | 36/50 |
| 0.12 / 0 / 0 | 49/50 | 50/50 | **50/50** | 50/50 |
| 0.12 / 1 / 1 | 44/50 | 50/50 | **50/50** | 41/50 |
| **total** | 292/400 | 348/400 | **397/400** | 368/400 |

Fix B's successes place the cube 3–11 mm from the target (median per setting) in 2.3–2.8 s (expert: 2.7 mm, 1.9 s). Smoothing hurt on 3 settings, so it is not used. Fine goal encoding (C) helped with the first, graded-only expert (3 → 15/50 on the week 7 setting) but added variance with the press-down one (38–50/50), so it is not used either.

**The control still holds:** with the new expert, a linear readout on the 8 noise-free signals (no brain) completes **0/50**, and an MLP **50/50**. So the fix doesn't make the task linear: any success still needs the nonlinearity the network supplies.

**The expert itself** (no brain): week 4 50/50, 3.8 mm, 2.0 s; graded press-down 50/50, 2.7 mm, 1.9 s; with 5 mm action noise 48/50 for both.

## This notebook
| step | what | GPU time |
|---|---|---|
| 1 | **Offline check, no brain time:** week 7's saved connectome data relabelled with the new expert; how well the same DN rates decode the new labels near the target | a few min, CPU |
| 2 | no-brain controls with the fix: expert, linear, MLP | ~3 min, CPU |
| 3 | **connectome** with the fix (week 7's encoder, w_scale, 80/80 timing, all-DN ridge readout, DAgger 4 × 30, same seeds) | ~30–45 min |
| 4 | random network with the fix (week 7's shuffle and w_scale) | ~30–45 min |
| 5–8 | comparison, verdict, plots, video, results file | ~5 min |

**Fixed before the run.** Pass: the connectome completes **≥ 40/50** test seeds. Success test unchanged (week 4 physics test: cube in the 5 cm zone, at rest, released, gripper 6 cm clear, held 0.4 s, within 6 s). Verdict unchanged from week 7 (connectome vs random network, milestone by milestone from success down, exact McNemar, p < 0.05; placement and time compared if both have ≥ 10 successes).

**If the connectome passes but the random network also does**, the conclusion is week 7's again: the brain's contribution comes from its network statistics, not its specific wiring. **If the connectome falls short of 40**, set `EXTRA_ROUNDS = 2` and Run all: DAgger continues from its saved checkpoint with 2 more brain-driven rounds, then retests (about 25 min per network).rerun with `FIX = 'B+D'` (smoothing; everything finished is loaded, only the networks rerun).

Set `QUICK = True` first for a ~5 min smoke test in `data/week8_quick/`. `RESUME = True` makes a disconnect cost at most one DAgger round.

### Setup (same as week 7)

In [ ]:
import sys, subprocess, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'brian2', 'numba', 'mujoco',
                        'imageio', 'imageio-ffmpeg'], check=True)
        print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
        os.kill(os.getpid(), 9)
    os.environ.setdefault('MUJOCO_GL', 'egl')
except ImportError:
    IN_COLAB = False

from pathlib import Path
BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain') if IN_COLAB else Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
ARM = next((p for p in [BRAIN.parent / 'arm', BRAIN.parent / 'week4/arm'] if (p / 'panda_env.py').exists()), None)
sys.path.insert(0, str(BRAIN))
if ARM: sys.path.insert(0, str(ARM))

QUICK = False        # True: smoke test (3 short DAgger rounds of 6 episodes, 10 test seeds), in data/week8_quick/
RESUME = True        # skip finished stages; DAgger resumes after its last finished round
FIX = 'B'          # the fix to run (see the table above): 'B', 'B+C', 'B+D', 'B+C+D', or 'week7' (no fix)
EXTRA_ROUNDS = 0     # fallback if the connectome falls short of 40/50: set 2 and rerun all. DAgger continues from
                     # its saved round-3 checkpoint (2 more brain-driven rounds), then the test is rerun
RUN_RANDOM = True    # Step 4: the random network with the same fix
USE_GPU = True
GPU_DTYPE = 'float32'
MAX_BATCH = 64

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = torch.cuda.get_device_name(0)
            print(f"GPU: {GPU}; torch {torch.__version__}")
        else:
            print("USE_GPU = True but no GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.")
    except ImportError:
        print("torch is not installed here -> numba on the CPU")

for p in ['sim/lif.py', 'sim/lif_numba.py', 'sim/interface.py', 'sim/readout.py', 'sim/closed_loop.py',
          'sim/pickplace.py', 'sim/network7.py', 'sim/batch7.py', 'sim/week7.py', 'sim/video7.py',
          'sim/expert8.py', 'sim/week8.py',
          'data/weights_male-cns_v1.0/meta.json', 'data/shortlist_week1.csv', 'data/wsyn_choice.json',
          'data/week7/week7_connectome_dagger.npz', 'data/week7/week7_connectome.pkl',
          'data/week7/week7_random_network_meta.json']:
    print(f"{p:50s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")
print(f"{'week 4 arm code':50s} {ARM if ARM else 'MISSING'}")

### Load the connectome, the neuron groups and the arm

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.pickplace import SIGNALS8, PP_MAX_STEPS
from sim.network7 import random_network
from sim.week7 import comparison_rows, verdict, paired_diff
from sim.week8 import FIXES, run_references8, run_network8, offline_relabel
from panda_env import PandaPickPlace, OBS_NAMES, TARGET_XY, ZONE_RADIUS

T_START = time.perf_counter()
HOST = 'colab' if IN_COLAB else 'laptop'
W7 = DATA / 'week7'
OUT = DATA / ('week8_quick' if QUICK else 'week8'); OUT.mkdir(exist_ok=True)

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
W_RAND = json.loads((W7 / 'week7_random_network_meta.json').read_text())['w_scale']   # week 7's activity match
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
full = ws.subnetwork(np.arange(ws.N))
ws._out = None; gc.collect()
env = PandaPickPlace()

N_ROUNDS = (3 if QUICK else 4) + EXTRA_ROUNDS
N_PER_ROUND = 6 if QUICK else 30
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],  # = week 7
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),
    test_seeds=list(range(92_000, 92_000 + (10 if QUICK else 50))),                                         # = week 7
    cut_seeds=list(range(92_000, 92_000 + (3 if QUICK else 10))),
    k=None, n_video=2, use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
print(f"male CNS: {ws.N:,} neurons; w_scale connectome {W}, random network {W_RAND:.3f} (week 7); "
      f"readout: all {len(ALL_DN)} DNs; fix {FIX}: {FIXES[FIX]}")
print(f"DAgger {N_ROUNDS} x {N_PER_ROUND} (week 7's seeds), test {len(CFG['test_seeds'])} seeds (week 7's); "
      f"results in {OUT}")

C = dict(blue='#2a78d6', orange='#eb6834', aqua='#1baf7a', yellow='#eda100', violet='#6250d6',
         red='#d03b3b', gray='#9a9893', ink='#0b0b0b', ink2='#52514e', grid='#e6e5e1')
POL_C = {'connectome': C['blue'], 'random network': C['orange'], 'MLP': C['aqua'],
         'linear on 8 signals': C['violet'], 'expert': C['gray'], 'connectome, week 7': '#9cc0ec'}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': C['grid'], 'axes.axisbelow': True, 'legend.frameon': False})

## Step 1: offline check on week 7's data (no brain time, a few minutes of CPU)

Week 7's connectome DAgger data has the DN rates at every state the arm visited. Here the same states are **relabelled with the graded expert**, and a readout is fitted (out-of-episode) to each expert's labels from the *same* rates. If the graded labels are easier to decode near the target, especially "go down" vs "stay", fix B should work on the real connectome. Smoothing 0.5 shows fix D on the same data.

In [ ]:
CONN7 = pickle.load(open(W7 / 'week7_connectome.pkl', 'rb'))
OFFLINE = offline_relabel(W7 / 'week7_connectome_dagger.npz', CONN7['pool_cols'], OBS_NAMES)
display(pd.DataFrame(OFFLINE).set_index(['labels', 'smoothing']))

## Step 2: the policies without a brain, with the same fix (a few minutes, physics only)

The expert (graded, if the fix includes B), the linear readout on the 8 noise-free signals and the MLP, trained with the same DAgger rounds and seeds, tested on the same 50 seeds. These are the controls: if the linear readout without a brain also succeeds with the fix, the fix made the task linear and the brain isn't what solved it.

In [ ]:
REF = run_references8(env, OBS_NAMES, CFG, OUT, FIX, resume=RESUME)
for k in ('expert', 'linear on 8 signals', 'MLP'):
    s = REF[k]['summary']
    print(f"{k:22s} success {s['success']}/{s['n']}, zone {s['zone_rate']:.0%}, placement(lifted) {s['place_mm_lifted']} mm, {s['outcomes']}")

## Step 3: the connectome with the fix (GPU: about 30–45 min)

Week 7's setup, frozen: encoder from the connectome's wiring (16 channels), w_scale from week 2, 80 ms of brain per 40 ms control step with an 80 ms window (the timing both networks had in the matched-timing check), readout = ridge on all DNs. DAgger 4 × 30 with week 7's seeds; test on week 7's 50 seeds; then the feedback-cut control. Checkpointed after every DAgger round.

In [ ]:
NETS = {}
NETS['connectome'] = run_network8('connectome', full, JO, ALL_DN, env, OBS_NAMES, CFG, OUT, w_scale=W, fix=FIX,
                                  resume=RESUME, extra_record=DN_cand)
s = NETS['connectome']['summary']
print(f"\nCONNECTOME [{FIX}]: success {s['success']}/{s['n']} (95% CI {s['ci'][0]:.0%}-{s['ci'][1]:.0%}), "
      f"placement {s['place_mm_success']} mm, time {s['time_s']} s, {s['outcomes']}")
print("while collecting, by round:", [f"{r['success']:.0%}" for r in NETS['connectome']['dagger']['log']])
print(f"peak memory {peak_mem_gb():.2f} GB")

## Step 4: the random network with the same fix (GPU: about 30–45 min)

Week 7's random network (every connection's target shuffled; degrees, weights and E/I kept), its w_scale from week 7's activity match, the same 80/80 timing, the same fix.

In [ ]:
if RUN_RANDOM:
    rnet = random_network(full, seed=0)
    NETS['random network'] = run_network8('random network', rnet, JO, ALL_DN, env, OBS_NAMES, CFG, OUT,
                                          w_scale=W_RAND, fix=FIX, resume=RESUME, extra_record=DN_cand)
    del rnet; gc.collect()
    s = NETS['random network']['summary']
    print(f"\nRANDOM NETWORK [{FIX}]: success {s['success']}/{s['n']}, placement {s['place_mm_success']} mm, {s['outcomes']}")

## Step 5: the comparison (deliverable)

**Pass for week 8 (fixed before the run): the connectome completes at least 40 of the 50 test seeds (80%).** Then week 7's pre-registered verdict (connectome vs random network, milestone by milestone, McNemar), now with success as the first milestone; placement and time are compared if both have at least 10 successes.

In [ ]:
SUMM = {k: v['summary'] for k, v in NETS.items()}
for k in ('MLP', 'linear on 8 signals', 'expert'):
    SUMM[k] = REF[k]['summary']
SUMM['connectome, week 7'] = CONN7['summary']
TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
display(TABLE.style.format({c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}, na_rep='-'))
TABLE.to_csv(OUT / f'week8_table_{HOST}.csv')

sc = SUMM['connectome']
PASS = sc['success'] >= 40
print(f"\nWEEK 8 PASS (connectome >= 40/50): {'YES' if PASS else 'NO'} ({sc['success']}/50)")
VERDICT = PM = PD = None
if 'random network' in NETS:
    VERDICT, PM = verdict(NETS['connectome']['outs'], NETS['random network']['outs'])
    print("VERDICT:", VERDICT)
    display(pd.DataFrame(PM).T.rename(columns={'a': 'connectome', 'b': 'random network',
                                               'only_a': 'only connectome', 'only_b': 'only random'}).round(4))
    PD = paired_diff(NETS['connectome']['outs'], NETS['random network']['outs'])
    print(f"success, connectome - random: {PD['diff']:+.0%} (paired bootstrap 95% CI {PD['ci'][0]:+.0%} to {PD['ci'][1]:+.0%})")
    sr = SUMM['random network']
    if sc['success'] >= 10 and sr['success'] >= 10:
        print(f"placement: connectome {sc['place_mm_success']} mm vs random {sr['place_mm_success']} mm; "
              f"time {sc['time_s']} s vs {sr['time_s']} s")
print("\nwhere each policy ended up:")
display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
for k in NETS:
    if 'cut' in NETS[k]:
        c = NETS[k]['cut']['summary']
        print(f"feedback cut, {k}: success {c['success']}/{c['n']}, lifted {c['lifted_rate']:.0%}")

## Step 6: what the policies did

In [ ]:
POLS = [k for k in ('connectome', 'random network', 'MLP', 'linear on 8 signals') if k in SUMM]
OUTS = {k: NETS[k]['outs'] for k in NETS}
OUTS.update({k: REF[k]['outs'] for k in ('MLP', 'linear on 8 signals', 'expert')})
OUTS['connectome, week 7'] = CONN7['outs']
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1.3, 1, 1]))
ax = axs[0]
ms = [('lifted_rate', 'lifted'), ('zone_rate', 'carried to zone'), ('success_rate', 'success')]
pp = ['connectome, week 7'] + POLS
wd = 0.8 / len(pp)
for j, p in enumerate(pp):
    ax.bar(np.arange(3) + (j - (len(pp) - 1) / 2) * wd, [SUMM[p][m] for m, _ in ms], wd * 0.92, color=POL_C[p], label=p)
ax.set_xticks(range(3), [l for _, l in ms]); ax.set_ylim(0, 1.05); ax.set_title(f'milestones (fix {FIX})')
ax.axhline(0.8, color=C['ink2'], lw=0.8, ls='--'); ax.legend(fontsize=7, loc='lower left')
ax = axs[1]
for p in [k for k in ('connectome', 'random network') if k in OUTS]:
    for o in OUTS[p]:
        cube = o['obs'][:, 8:10] * 1000
        ax.plot(cube[:, 0], cube[:, 1], color=POL_C[p], lw=0.7, alpha=0.5)
    ax.plot([], [], color=POL_C[p], label=p)
ax.add_patch(plt.Circle(TARGET_XY * 1000, ZONE_RADIUS * 1000, fill=False, color=C['ink'], lw=1.2))
ax.set_aspect('equal'); ax.set_xlabel('cube x (mm)'); ax.set_ylabel('cube y (mm)'); ax.set_title('cube paths')
ax.legend(fontsize=7, loc='lower left')
ax = axs[2]
for j, p in enumerate(pp + ['expert']):
    v = [o['place_error_mm'] for o in OUTS[p] if o['lifted']]
    ax.scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.18, 0.18, len(v)), v, s=9, color=POL_C[p], alpha=0.7)
ax.axhline(ZONE_RADIUS * 1000, color=C['ink2'], lw=0.8, ls='--')
ax.set_yscale('log'); ax.set_xticks(range(len(pp) + 1), [p.replace(' on 8 signals', '') for p in pp + ['expert']], rotation=25)
ax.set_ylabel('cube to target at the end (mm)'); ax.set_title('placement error (lifted episodes)')
fig.tight_layout(); fig.savefig(OUT / f'week8_summary_{HOST}.png', bbox_inches='tight'); plt.show()

## Step 7: the video

Each network's best two test episodes and its first failure, if any (recorded during the test). Rendered with week 7's `video7.render`.

In [ ]:
EPISODES = [e for k in NETS for e in NETS[k]['video_eps']]
for e in EPISODES:
    print(f"{e['net']}, seed {e['seed']}: {e['outcome']}")
pickle.dump(EPISODES, open(OUT / 'week8_episodes.pkl', 'wb'))
from sim.video7 import render
VIDEO = OUT / f'week8_pick_place_{HOST}.mp4'
GL = render(str(OUT / 'week8_episodes.pkl'), str(VIDEO), slow=0.5)
if GL:
    from IPython.display import Video
    display(Video(str(VIDEO), embed=True, width=900))

## Step 8: results file

In [ ]:
summary = dict(host=HOST, quick=QUICK, gpu=GPU, fix=FIX, fix_settings={k: str(v) for k, v in FIXES[FIX].items()},
               offline=OFFLINE, table=TABLE.reset_index().to_dict('records'), passed=PASS, verdict=VERDICT,
               paired_milestones=PM, paired_success=PD, outcomes={k: s['outcomes'] for k, s in SUMM.items()},
               networks={k: dict(dagger_log=v['dagger']['log'], alpha=v['dagger']['alpha'],
                                 cut=v['cut']['summary'] if 'cut' in v else None, wall_min=v['wall_min'],
                                 brain_s_per_sim_s=v.get('brain_s_per_sim_s')) for k, v in NETS.items()},
               wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week8_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)))
print(f"this session {summary['wall_min']:.0f} min. Send me week8_results_{HOST}.json, week8_summary_{HOST}.png and the video from {OUT}.")